### **Package**

In [ ]:
import subprocess
import sys
import os

print(sys.version)

packages = {
    "pymoo": "pymoo",
    "GPy": "GPy",
    "autogluon.tabular": "autogluon.tabular",
    "yaml": "PyYAML",
}

for module_name, pip_name in packages.items():
    try:
        __import__(module_name)
        print(f"{module_name} is already installed.")
    except ImportError:
        print(f"{module_name} is not installed. Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        print(f"{pip_name} installed.")

import warnings
warnings.filterwarnings("ignore", message=".*load_learner.*pickle.*")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/PhD 2026-1 New metrics')

from src.opt_problem import build_problem, Benchmark_Problem, EvaluatePreRealCallback, evaluate_pre_real
from src.survival import Survival_standard, Survival_dual_ranking
from src.data import generate_data
from src.metrics import get_metrics
from src.models import GPR_RBF, gpr_pred_mean_std
from src.uncertainty import coverage, find_alpha
from src.experiment import run_experiment
from src.other_functions import mean_std, print_gpr_params
from src.plotting import plot_z_score, plot_obj_2d, plot_y_true_pred

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.lhs import LHS
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from sklearn.metrics import mean_squared_error
from pathlib import Path
import csv
import yaml
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt


### **Main**

###### 1. Initial settings

In [ ]:
# Problem and experiment settings are loaded from YAML so they are easy to edit.
config_path = Path("configs/exp1_gpr_rbf.yaml")
if not config_path.exists():
    raise FileNotFoundError(f"Cannot find config file: {config_path.resolve()}")

with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

problem_name = config.get("problem_name", "dtlz1")
n_var = config.get("n_var", 10)
n_obj = config.get("n_obj", 2)
n_gen = config.get("n_gen", 100)
pop_size = config.get("pop_size", 100)
n_runs = config.get("n_runs", 30)
ea_seed_start = config.get("ea_seed_start", 1)
ea_seed_end = config.get("ea_seed_end", 31)
ea_seeds = range(ea_seed_start, ea_seed_end)
no_plot = config.get("no_plot", False)
run_ea = config.get("run_ea", False)
run_bias_variance_ea = config.get("run_bias_variance_ea", True)
output_dir = Path(config.get("output_dir", f"outputs/exp1_gpr_rbf/{problem_name}"))
output_dir.mkdir(parents=True, exist_ok=True)
bias_variance_configs = config["bias_variance_configs"]

problem = build_problem(
    problem_name=problem_name,
    n_var=n_var,
    n_obj=n_obj)
print(f"Problem name: {problem_name}")
print(f"Cons: {problem.n_constr}")
print(f"Var: {n_var}")
print(f"Obj: {n_obj}")
print(f"Output dir: {output_dir}")

# Metrics
hv, igd_plus, obj_min, obj_max, ref_point = get_metrics(
    problem_name=problem_name,
    problem=problem,
    n_var=n_var,
    n_obj=n_obj)
print('\nMin-Max normalization -> Min: ', obj_min)
print('Min-Max normalization -> Max: ', obj_max)
print('HV Reference points: ', ref_point)

np.set_printoptions(precision=4, suppress=True)

# Data: LHS sampling with seed 42
random_seed = 42
sampling = LHS()
sample_size = 11 * n_var - 1
X_train, y_train, X_val, y_val, X_test, y_test = generate_data(
    problem=problem,
    sample_size=sample_size,
    sampling=sampling,
    train_seed=random_seed,
    val_size=100,
    test_size=100,
    test_seed=1)

y_train_f1 = y_train[:, 0]
y_train_f2 = y_train[:, 1]
print(f"Sampling X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"y_test shape: {y_test.shape}")

if not no_plot:
    plot_obj_2d(y_train,
                xlim=(-100, 700),
                ylim=(-100, 700))


###### 2. Surrogate model training (normal)

In [ ]:
model_f1 = GPR_RBF()
model_f2 = GPR_RBF()

model_f1.fit(X_train, y_train_f1)
model_f2.fit(X_train, y_train_f2)
print_gpr_params(model_f1, model_f2)

# Predict
pred_mean, pred_std, mean_f1, std_f1, mean_f2, std_f2 = gpr_pred_mean_std(
    model_f1,
    model_f2,
    X_test,
    noiseless=True)
print(f"\nGPR(RBF) MSE: {mean_squared_error(y_test, pred_mean):.2e}\n")
plot_y_true_pred(y_test,pred_mean)

###### 2.1 Surrogate model training for six bias/variance settings (seed=42)

In [ ]:
def train_weighted_gpr_pair(X_train, y_train, X_test, y_test, lengthscale_weight, plot=True):
    model_f1 = GPR_RBF()
    model_f2 = GPR_RBF()
    model_f1.fit_high_bias_variance(X_train, y_train[:, 0], lengthscale_weight=lengthscale_weight)
    model_f2.fit_high_bias_variance(X_train, y_train[:, 1], lengthscale_weight=lengthscale_weight)
    print_gpr_params(model_f1, model_f2)

    pred_mean, pred_std, _, _, _, _ = gpr_pred_mean_std(model_f1, model_f2, X_test, noiseless=True)
    test_mse = mean_squared_error(y_test, pred_mean)
    print(f"\nGPR(RBF) MSE: {test_mse:.2e}")

    if plot:
        plot_y_true_pred(y_test, pred_mean)

    return model_f1, model_f2, pred_mean, pred_std, test_mse


seed42_models_by_type = {}

for bv_config in bias_variance_configs:
    model_type = bv_config["model_type"]
    lengthscale_weight = bv_config["lengthscale_weight"]
    print(f"\n=== {model_type} model (lengthscale_weight={lengthscale_weight:g}) ===")

    model_f1_bv, model_f2_bv, pred_mean_bv, pred_std_bv, test_mse_bv = train_weighted_gpr_pair(
        X_train,
        y_train,
        X_test,
        y_test,
        lengthscale_weight=lengthscale_weight,
        plot=not no_plot)

    seed42_models_by_type[model_type] = {
        "model_f1": model_f1_bv,
        "model_f2": model_f2_bv,
        "lengthscale_weight": lengthscale_weight,
        "seed42_test_mse": test_mse_bv,
    }


### Bias-Variance Experiment (30 samplings / 30 trainings)

In [ ]:
def write_rows_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def run_bias_variance_experiment(lengthscale_weight, model_type, n_runs=30):
    preds = []
    y_test_ref = None
    training_records = []

    for seed in range(1, n_runs + 1):
        X_train_i, y_train_i, _, _, X_test_i, y_test_i = generate_data(
            problem=problem,
            sample_size=sample_size,
            sampling=LHS(),
            train_seed=seed,
            val_size=100,
            test_size=100,
            test_seed=1)

        model_f1_i = GPR_RBF()
        model_f2_i = GPR_RBF()
        model_f1_i.fit_high_bias_variance(
            X_train_i, y_train_i[:, 0], lengthscale_weight=lengthscale_weight)
        model_f2_i.fit_high_bias_variance(
            X_train_i, y_train_i[:, 1], lengthscale_weight=lengthscale_weight)

        pred_mean_i, _, _, _, _, _ = gpr_pred_mean_std(
            model_f1_i, model_f2_i, X_test_i, noiseless=True)
        preds.append(pred_mean_i)

        training_records.append({
            "model_type": model_type,
            "run": seed,
            "train_seed": seed,
            "lengthscale_weight": lengthscale_weight,
            "test_mse": mean_squared_error(y_test_i, pred_mean_i),
            "f1_lengthscale": np.array2string(model_f1_i.model.kern.lengthscale.values, precision=8, separator=" "),
            "f1_kernel_variance": float(model_f1_i.model.kern.variance.values[0]),
            "f1_noise": float(model_f1_i.model.Gaussian_noise.variance.values[0]),
            "f2_lengthscale": np.array2string(model_f2_i.model.kern.lengthscale.values, precision=8, separator=" "),
            "f2_kernel_variance": float(model_f2_i.model.kern.variance.values[0]),
            "f2_noise": float(model_f2_i.model.Gaussian_noise.variance.values[0]),
        })

        if y_test_ref is None:
            y_test_ref = y_test_i

    preds = np.array(preds)
    mean_pred = preds.mean(axis=0)
    bias = np.mean((mean_pred - y_test_ref) ** 2)
    variance = np.mean(np.var(preds, axis=0))
    return bias, variance, training_records


bias_variance_summary_rows = []
bias_variance_training_rows = []

for bv_config in bias_variance_configs:
    model_type = bv_config["model_type"]
    lengthscale_weight = bv_config["lengthscale_weight"]
    bias, variance, training_records = run_bias_variance_experiment(
        lengthscale_weight=lengthscale_weight,
        model_type=model_type,
        n_runs=n_runs)

    print(f"{model_type} model -> Bias: {bias:.6e}, Variance: {variance:.6e}")

    seed42_models_by_type[model_type]["bias"] = bias
    seed42_models_by_type[model_type]["variance"] = variance
    bias_variance_training_rows.extend(training_records)
    bias_variance_summary_rows.append({
        "problem_name": problem_name,
        "model_type": model_type,
        "n_runs": n_runs,
        "lengthscale_weight": lengthscale_weight,
        "bias": bias,
        "variance": variance,
    })

write_rows_csv(output_dir / "bias_variance_training_records.csv", bias_variance_training_rows)
write_rows_csv(output_dir / "bias_variance_summary.csv", bias_variance_summary_rows)

pd.DataFrame(bias_variance_summary_rows)


###### CICP & Z score; find_alpha

In [ ]:
###### CICP
"""
Interval    Coverage
±1.282σ      80.00%
±1.645σ      90.00%
±1.960σ      95.00%
"""

for k in [1.645]:
    per_dim, overall = coverage(y_test, pred_mean, pred_std, k=k)
    print(f"k={k}: per_dim={per_dim*100}%, overall={overall*100:.1f}%")

###### Z score
plot_z_score(y_test, pred_mean, pred_std)

alpha_c90_f1 = find_alpha(
    X_val, y_val[:,0],
    model_f1,
    target_coverage=0.9)
alpha_c90_f2 = find_alpha(
    X_val, y_val[:,1],
    model_f2,
    target_coverage=0.9)
print(f"alpha_c90_f1={alpha_c90_f1:.2f}, alpha_c90_f2={alpha_c90_f2:.2f}")

F_upper_c90 = np.stack(
    [mean_f1 + alpha_c90_f1 * std_f1,
     mean_f2 + alpha_c90_f2 * std_f2], axis=1)
print('F_upper_c90\n', F_upper_c90[0:5],'\n')

###### 3. Optimization for seed=42 bias/variance models

In [ ]:
def summarize_ea_results(results):
    mean_mse, std_mse = mean_std(results["mse_list"])
    mean_igd, std_igd = mean_std(results["igd_list"])
    mean_hv_real, std_hv_real = mean_std(results["hv_real_list"])
    mean_hv_surrogate, std_hv_surrogate = mean_std(results["hv_surrogate_list"])

    return {
        "mse_mean": mean_mse,
        "mse_std": std_mse,
        "igd_plus_mean": mean_igd,
        "igd_plus_std": std_igd,
        "hv_surrogate_mean": mean_hv_surrogate,
        "hv_surrogate_std": std_hv_surrogate,
        "hv_real_mean": mean_hv_real,
        "hv_real_std": std_hv_real,
    }


def print_ea_summary(summary):
    print(f"MSE: Mean = {summary['mse_mean']:.2e}, Std = {summary['mse_std']:.2e}")
    print(f"IGD+: Mean = {summary['igd_plus_mean']:.2e}, Std = {summary['igd_plus_std']:.2e}")
    print(f"Sur HV: Mean = {summary['hv_surrogate_mean']:.2f}, Std = {summary['hv_surrogate_std']:.2f}")
    print(f"Real HV: Mean = {summary['hv_real_mean']:.2f}, Std = {summary['hv_real_std']:.2f}")


optimization_summary_rows = []

if run_bias_variance_ea:
    for model_type, model_info in seed42_models_by_type.items():
        print(f"\n=== EA for {problem_name} | seed=42 {model_type} model ===")
        results = run_experiment(
            problem=problem,
            problem_name=problem_name,
            n_gen=n_gen,
            pop_size=pop_size,
            model_f1=model_info["model_f1"],
            model_f2=model_info["model_f2"],
            obj_min=obj_min,
            obj_max=obj_max,
            hv=hv,
            igd_plus=igd_plus,
            use_surrogate="GPR_uncertainty",
            survival_function=Survival_standard(),
            use_callback=False,
            seeds=ea_seeds)

        summary = summarize_ea_results(results)
        print_ea_summary(summary)
        optimization_summary_rows.append({
            "problem_name": problem_name,
            "model_type": model_type,
            "train_seed": 42,
            "lengthscale_weight": model_info["lengthscale_weight"],
            "bias": model_info["bias"],
            "variance": model_info["variance"],
            **summary,
        })

        write_rows_csv(
            output_dir / "optimization_summary_seed42_bias_variance_models.csv",
            optimization_summary_rows)
else:
    print("run_bias_variance_ea is false; skipping bias/variance model optimization.")


In [ ]:
optimization_summary_df = pd.DataFrame(optimization_summary_rows)
optimization_summary_df


###### 3. Optimization GPR (RBF) + dual-ranking (c=0.90)

In [ ]:
results = run_experiment(
    problem=problem,
    problem_name=problem_name,
    n_gen=n_gen,
    pop_size=pop_size,
    model_f1=model_f1,
    model_f2=model_f2,
    obj_min=obj_min,
    obj_max=obj_max,
    hv=hv,
    igd_plus=igd_plus,
    use_surrogate="GPR_uncertainty", ### change
    survival_function=Survival_dual_ranking(
        alpha_f1=alpha_c90_f1,
        alpha_f2=alpha_c90_f2), ### change
    use_callback=False, ### change
    seeds=range(1, 2)) ### change

mse_list = results["mse_list"]
igd_list = results["igd_list"]
hv_surrogate_list = results["hv_surrogate_list"]
hv_real_list = results["hv_real_list"]

obj = results["run_details"][-1]["obj"]
f_real = results["run_details"][-1]["f_real"]
solution = results["run_details"][-1]["solution"]

plot_obj_2d(obj,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))
plot_obj_2d(f_real,
            xlim=(float(obj_min[0])-0.3*float(obj_max[0]), float(obj_max[0])),
            ylim=(float(obj_min[1])-0.3*float(obj_max[1]), float(obj_max[1])))


In [ ]:
# mean_mse, std_mse = mean_std(mse_list)
# mean_igd, std_igd = mean_std(igd_list)
# mean_hv_real, std_hv_real = mean_std(hv_real_list)
# mean_hv_surrogate, std_hv_surrogate = mean_std(hv_surrogate_list)

# print('Problem name: ', problem_name)
# print("\n=== GPR + dual-ranking + alpha_c90  ===")

# print(f"MSE: Mean = {mean_mse:.2e}, Std = {std_mse:.2e}")
# print(f"IGD+: Mean = {mean_igd:.2e}, Std = {std_igd:.2e}")
# print(f"Sur HV: Mean = {mean_hv_surrogate:.2f}, Std = {std_hv_surrogate:.2f}")
# print(f"Real HV: Mean = {mean_hv_real:.2f}, Std = {std_hv_real:.2f}")